# Hari 18 — Evaluasi Formal & Kesesuaian dengan Business Success Criteria

**Skor sejauh ini (setelah koreksi Hari 17):**
- Baseline naive: MAE 12.953
- Linear Regression (Hari 16): MAE test **7.806,56** ← masih juara
- Random Forest (Hari 17): MAE test **8.759,11**

Sejauh ini kita cuma bandingkan dari sisi **teknis** (MAE/RMSE). Tapi ingat Project Charter Hari 2 — ada juga **business success criteria** yang belum pernah benar-benar diuji: *"output prediksi bisa diringkas jadi label sederhana (↑ Naik / ↓ Turun / → Stabil) yang bisa dipahami tanpa latar belakang statistik"*. Hari ini kita bangun itu, sekaligus tabel perbandingan formal.

In [6]:
# Cell ini sudah lengkap — muat ulang data & fit ulang kedua model (notebook ini mandiri).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = pd.read_csv("X_train.csv", index_col=0, parse_dates=True)
X_test = pd.read_csv("X_test.csv", index_col=0, parse_dates=True)
X_train_scaled = pd.read_csv("X_train_scaled.csv", index_col=0, parse_dates=True)
X_test_scaled = pd.read_csv("X_test_scaled.csv", index_col=0, parse_dates=True)
y_train = pd.read_csv("y_train.csv", index_col=0, parse_dates=True).iloc[:, 0]
y_test = pd.read_csv("y_test.csv", index_col=0, parse_dates=True).iloc[:, 0]

model_linear = LinearRegression().fit(X_train_scaled, y_train)
model_rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)

pred_test_linear = model_linear.predict(X_test_scaled)
pred_test_rf = model_rf.predict(X_test)

print("Kedua model berhasil dilatih ulang.")

Kedua model berhasil dilatih ulang.


## Tabel Perbandingan Formal (Kriteria Teknis)

In [12]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Hitung MAE dan RMSE test untuk model_linear (pred_test_linear vs y_test)
# 2. Hitung MAE dan RMSE test untuk model_rf (pred_test_rf vs y_test)
# 3. Buat dictionary/DataFrame dengan baris ["Baseline", "Linear Regression", "Random Forest"]
#    dan kolom ["MAE Test", "RMSE Test", "Perbaikan vs Baseline (%)"]
#    - Baseline: MAE Test = 12953, RMSE Test = None (baseline tidak pernah dihitung RMSE-nya
#      di Hari 2, boleh dikosongkan dengan np.nan), Perbaikan = 0%
#    - Linear Regression & Random Forest: isi dari hasil hitungan di atas, hitung
#      perbaikan dengan (12953 - mae_test) / 12953 * 100
# 4. Buat pd.DataFrame dari dictionary itu, cetak tabelnya
# Tulis kode kamu di bawah ini:
mae_model_linear = mean_absolute_error(y_test, pred_test_linear)
rmse_model_linier = mean_squared_error(y_test, pred_test_linear) ** 0.5
mae_model_rf = mean_absolute_error(y_test, pred_test_rf)
rmse_model_rf = mean_squared_error(y_test, pred_test_rf) ** 0.5

baseline_mae = 12953
perbaikan_linear = (baseline_mae - mae_model_linear) / baseline_mae * 100
perbaikan_rf = (baseline_mae - mae_model_rf) / baseline_mae * 100
data = {
    "MAE Test":[baseline_mae, mae_model_linear, mae_model_rf],
    "RMSE Test":[None, rmse_model_linier, rmse_model_rf],
    "Perbaikan vs Baseline (%)":[0, perbaikan_linear, perbaikan_rf]
}

perbandingan_model = pd.DataFrame(data, index=["Baseline", "Linear Regression", "Random Forest"])
print(perbandingan_model)

                       MAE Test     RMSE Test  Perbaikan vs Baseline (%)
Baseline           12953.000000           NaN                   0.000000
Linear Regression   7806.556548   9975.024501                  39.731672
Random Forest       8759.106897  11425.891717                  32.377774


**Kriteria teknis dari Project Charter Hari 2:** MAE test harus di bawah baseline, target realistis perbaikan minimal 15-20%. Cek tabel di atas — kedua model kemungkinan besar sudah lolos kriteria ini.

## Membangun Label Tren (Kriteria Bisnis)

Business objective dari Hari 2: masyarakat awam butuh tahu apakah kasus **naik, turun, atau stabil** minggu depan — bukan angka mentah. Untuk mengubah prediksi jadi label seperti itu, kita perlu **nilai pembanding**: kasus minggu ini (yang sudah diketahui) dibandingkan dengan prediksi minggu depan.

Masalahnya: kasus minggu ini (`kasus_baru_mingguan` pada tanggal yang sama dengan baris `X_test`) **tidak pernah dimasukkan sebagai fitur** — fitur paling baru yang ada cuma `lag_1` (kasus SATU minggu sebelum tanggal baris itu). Jadi kita perlu ambil ulang angka aslinya dari `dataset_bersih_minggu2.csv` (hasil Hari 9).

In [13]:
# Cell ini sudah lengkap.
kasus_asli = pd.read_csv("dataset_bersih_minggu2.csv", index_col=0, parse_dates=True)["kasus_baru_mingguan"]

# Ambil kasus MINGGU INI (nilai yang sudah diketahui) untuk setiap baris di X_test
current_actual_test = kasus_asli.loc[X_test.index]

print("Contoh 5 baris pertama:")
print(pd.DataFrame({
    "kasus_minggu_ini (diketahui)": current_actual_test,
    "kasus_minggu_depan (target aktual)": y_test
}).head())

Contoh 5 baris pertama:
                           kasus_minggu_ini (diketahui)  \
Date                                                      
2022-06-05 00:00:00+00:00                        2385.0   
2022-06-12 00:00:00+00:00                        3688.0   
2022-06-19 00:00:00+00:00                        7587.0   
2022-06-26 00:00:00+00:00                       12376.0   
2022-07-03 00:00:00+00:00                       13466.0   

                           kasus_minggu_depan (target aktual)  
Date                                                           
2022-06-05 00:00:00+00:00                              3688.0  
2022-06-12 00:00:00+00:00                              7587.0  
2022-06-19 00:00:00+00:00                             12376.0  
2022-06-26 00:00:00+00:00                             13466.0  
2022-07-03 00:00:00+00:00                             17388.0  


In [14]:
# Cell ini sudah lengkap — fungsi untuk mengubah angka jadi label tren.

def label_tren(nilai_depan, nilai_sekarang, ambang=0.05):
    """Bandingkan nilai_depan terhadap nilai_sekarang, kembalikan label tren.
    ambang=0.05 artinya perubahan di bawah 5% dianggap 'Stabil'."""
    perubahan = (nilai_depan - nilai_sekarang) / nilai_sekarang
    if perubahan > ambang:
        return "Naik"
    elif perubahan < -ambang:
        return "Turun"
    else:
        return "Stabil"

# Label tren AKTUAL (dari data sebenarnya)
tren_aktual = [label_tren(y_test.iloc[i], current_actual_test.iloc[i]) for i in range(len(y_test))]

print("Distribusi label tren aktual di test set:")
print(pd.Series(tren_aktual).value_counts())

Distribusi label tren aktual di test set:
Naik      14
Turun     13
Stabil     2
Name: count, dtype: int64


## Latihan: Hitung Trend Accuracy untuk Kedua Model

In [15]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat tren_prediksi_linear: list hasil label_tren(pred_test_linear[i], current_actual_test.iloc[i])
#    untuk setiap i dalam range(len(pred_test_linear)) — sama polanya seperti tren_aktual di atas
# 2. Buat tren_prediksi_rf dengan cara yang sama, tapi pakai pred_test_rf
# 3. Hitung trend_accuracy_linear = proporsi elemen tren_prediksi_linear yang SAMA dengan
#    elemen tren_aktual di posisi yang sama (bisa pakai perbandingan list lalu np.mean(),
#    atau ubah dulu keduanya jadi pd.Series lalu bandingkan dengan ==)
# 4. Hitung trend_accuracy_rf dengan cara yang sama
# 5. Cetak kedua trend accuracy dalam bentuk persentase, dan kesimpulan: model mana yang
#    lebih baik dari sisi LABEL TREN (belum tentu sama dengan yang menang di MAE!)
# Tulis kode kamu di bawah ini:
tren_prediksi_linear = [label_tren(pred_test_linear[i], current_actual_test.iloc[i]) 
                        for i in range(len(pred_test_linear))
]
tren_prediksi_rf = [label_tren(pred_test_rf[i], current_actual_test.iloc[i])
                    for i in range(len(pred_test_rf))]
tren_accuracy_linear = (pd.Series(tren_prediksi_linear) == pd.Series(tren_aktual)).mean()
tren_accuracy_rf = (pd.Series(tren_prediksi_rf) == pd.Series(tren_aktual)).mean()

print(f"Tren Accuracy Linear Regression: {tren_accuracy_linear * 100:.2f}%")
print(f"Tren Accuracy Random Forest: {tren_accuracy_rf * 100:.2f}%")
if tren_accuracy_linear > tren_accuracy_rf:
    print("Linear Regression lebih baik dari sisi label tren")
else:
    print("Random Forest lebih baik dari sisi label tren")


Tren Accuracy Linear Regression: 62.07%
Tren Accuracy Random Forest: 37.93%
Linear Regression lebih baik dari sisi label tren


## Lihat Detail Kesalahan Label (Opsional tapi Insightful)

In [16]:
# Cell ini sudah lengkap — tabel silang (confusion matrix sederhana) untuk Linear Regression:
# baris = label aktual, kolom = label prediksi. Angka di luar diagonal = kesalahan.
# (Butuh variabel tren_prediksi_linear dari cell latihan sebelumnya.)

tabel_silang = pd.crosstab(
    pd.Series(tren_aktual, name="Aktual"),
    pd.Series(tren_prediksi_linear, name="Prediksi (Linear Regression)")
)
print(tabel_silang)

Prediksi (Linear Regression)  Naik  Stabil  Turun
Aktual                                           
Naik                            14       0      0
Stabil                           1       1      0
Turun                            6       4      3


## Sintesis Akhir: Kriteria Teknis vs Kriteria Bisnis

Sekarang kamu punya dua sudut pandang untuk menilai model:

| | Kriteria Teknis (MAE) | Kriteria Bisnis (Trend Accuracy) |
|---|---|---|
| Linear Regression | 7.806,56 | 62.07% |
| Random Forest | 8.759,11 | 37.93% |

**Pertanyaan pentingnya:** apakah model yang menang di MAE (angka presisi) otomatis juga menang di trend accuracy (arah yang dipahami masyarakat)? Belum tentu — model bisa saja punya MAE lebih rendah tapi sesekali salah arah di titik kritis (misal pas transisi dari naik ke turun), atau sebaliknya.

## Catatan untuk Pengembangan Lanjutan (Opsional, Tidak Wajib Sekarang)

Saat menyiapkan `current_actual_test` di atas, kita jadi sadar satu hal: **"kasus minggu ini" (nilai paling baru yang sebenarnya paling informatif) tidak pernah dimasukkan sebagai fitur ke model** — fitur paling baru yang ada cuma `lag_1` (mundur 1 minggu). Ini sering disebut fitur "lag-0" yang terlewat. Kalau nanti kamu mau eksperimen lanjutan (di luar 30 hari roadmap ini), coba tambahkan fitur ini dan lihat apakah MAE membaik — kemungkinan besar iya, karena ini adalah informasi paling dekat waktunya dengan target yang diprediksi.

## Refleksi Hari 18

> 1. Berapa trend accuracy Linear Regression vs Random Forest? Apakah urutannya sama dengan urutan MAE? → **62.07 vs 37.93%, Ya**
> 2. Dari tabel silang (confusion matrix) di atas — kesalahan paling sering terjadi di label apa? (misal: aktual "Naik" tapi diprediksi "Stabil") → **aktual turun tapi model bilang naik**
> 3. Kalau kamu harus pilih SATU model final untuk proyek ini, mana yang kamu pilih — dan apakah pertimbangannya lebih ke MAE atau trend accuracy? → Linear Regression, pertimbangan lebih ke arah tend accuracy

---
### Selanjutnya: Hari 19-21

- **Hari 19-20**: Cross-validation yang benar untuk time-series (`TimeSeriesSplit`, bukan `KFold`), dan kalau Random Forest masih ingin dicoba diperbaiki, eksperimen `max_depth`/`min_samples_leaf`
- **Hari 21**: Mini-project penutup Minggu 3 — kunci model final berdasarkan gabungan kriteria teknis + bisnis dari hari ini, siap lanjut ke Evaluation & Deployment (Minggu 4)